[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/55_multi_agent_solution.ipynb)

#  Solution: Multi-Agent Message Router

Reference: AutoGen (Microsoft), CrewAI, LangGraph multi-agent patterns, FIPA ACL agent communication standard.

```text
1. Parse message destination: single agent, list, or "*"
2. Single: call agent[to](message)
3. List: iterate and call each
4. Broadcast: call all agents except sender
5. Type routing: filter recipients by routing_rules[type]
```

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
#  SOLUTION

def agent_message_router(message, agent_registry, routing_rules=None):
    """Route messages between agents.

    Inspired by:
    - AutoGen (Microsoft): structured agent-to-agent messaging
    - CrewAI: role-based agent delegation
    - LangGraph: graph-based agent orchestration
    - FIPA ACL: standard agent communication (inform, request, query, etc.)

    In production multi-agent systems (SWE-agent, ChemCrow, MetaGPT),
    message routing evolves into a full pub-sub or graph-execution pattern.
    """
    sender = message.get("from", "")
    recipients = message["to"]
    msg_type = message.get("type", "")

    # Resolve recipients
    if isinstance(recipients, str):
        if recipients in ("*", "broadcast"):
            recipient_list = [name for name in agent_registry if name != sender]
        else:
            recipient_list = [recipients]
    elif isinstance(recipients, list):
        recipient_list = recipients
    else:
        return []

    # Apply routing rules if present
    if routing_rules and msg_type in routing_rules:
        allowed = routing_rules[msg_type]
        recipient_list = [r for r in recipient_list if r in allowed]

    # Deliver messages and collect responses
    responses = []
    for name in recipient_list:
        if name in agent_registry:
            try:
                response = agent_registry[name](message)
                responses.append(response)
            except Exception:
                responses.append({"from": name, "error": "agent_error"})

    return responses

In [ ]:
#  Verify
agents = {
    "a": lambda m: {"from": "a", "status": "ok"},
    "b": lambda m: {"from": "b", "status": "ok"},
}
r = agent_message_router({"from": "sys", "to": "*", "content": "ping"}, agents)
print(f"Broadcast: {len(r)} responses from {[x['from'] for x in r]}")

In [ ]:
from torch_judge import check
check("multi_agent")